<a href="https://colab.research.google.com/github/angelo-chu/python/blob/main/Binance_kline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
import zipfile

uploaded = files.upload()

Saving BTCUSDT-1d-2026-09.zip to BTCUSDT-1d-2026-09.zip


In [3]:
filename = "BTCUSDT-1d-2026-09.zip"

with zipfile.ZipFile(filename) as archive:
    print("Files inside:", archive.namelist())

    csv_name = next(
        name for name in archive.namelist()
        if name.endswith(".csv")
    )

    with archive.open(csv_name) as csv_file:
        for _ in range(3):
            print(csv_file.readline().decode().strip())

Files inside: ['BTCUSDT-1d-2026-09.csv']
1788220800000000,78581.30000000,79220.61000000,76420.00000000,77439.00000000,14130.63949000,1788307199999999,1101387276.65132930,3662805,7128.14511000,555981558.22935560,0
1788307200000000,77439.01000000,77792.00000000,76264.00000000,77340.01000000,13070.97820000,1788393599999999,1008333112.58752340,3312492,6647.38296000,512773293.69380870,0
1788393600000000,77340.01000000,82300.00000000,76968.00000000,81270.37000000,19806.42247000,1788479999999999,1579639079.84716540,4460645,10841.35467000,864605005.48359890,0


In [4]:
import pandas as pd

columns = [
    "open_time_us",
    "open",
    "high",
    "low",
    "close",
    "base_volume",
    "close_time_us",
    "quote_volume",
    "trade_count",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
]

with zipfile.ZipFile(filename) as archive:
    df = pd.read_csv(
        archive.open(csv_name),
        header=None,
        names=columns
    )

# The timestamps shown in your file are in microseconds.
df["open_time_utc"] = pd.to_datetime(
    df["open_time_us"], unit="us", utc=True
)

display(df[[
    "open_time_utc",
    "open",
    "high",
    "low",
    "close",
    "base_volume",
    "quote_volume",
    "trade_count"
]].head())

print("Rows:", len(df))
print("First date:", df["open_time_utc"].min())
print("Last date:", df["open_time_utc"].max())
print("Duplicate opening timestamps:",
      df["open_time_us"].duplicated().sum())

,open_time_utc,open,high,low,close,base_volume,quote_volume,trade_count
0,2026-09-01 00:00:00+00:00,78581.30,79220.61,76420.0,77439.00,14130.63949,1.101387e+09,3662805
1,2026-09-02 00:00:00+00:00,77439.01,77792.00,76264.0,77340.01,13070.97820,1.008333e+09,3312492
2,2026-09-03 00:00:00+00:00,77340.01,82300.00,76968.0,81270.37,19806.42247,1.579639e+09,4460645
3,2026-09-04 00:00:00+00:00,81270.37,81427.75,78660.0,79660.77,19198.62549,1.538393e+09,3417900
4,2026-09-05 00:00:00+00:00,79660.77,80200.00,79442.0,79831.75,9117.24020,7.271621e+08,1276050


Rows: 30
First date: 2026-09-01 00:00:00+00:00
Last date: 2026-09-30 00:00:00+00:00
Duplicate opening timestamps: 0


In [5]:
import sqlite3

# Create the SQLite database.
conn = sqlite3.connect("/content/exchange_analysis.db")

# Prepare the data without changing your original df.
daily = df.copy()

daily["open_time_utc"] = daily["open_time_utc"].dt.strftime(
    "%Y-%m-%d %H:%M:%S"
)
daily["symbol"] = "BTCUSDT"

# Load the data into a SQLite table.
daily.to_sql(
    "candles_daily",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("Created candles_daily.")

Created candles_daily.


In [6]:
%pip install jupysql --quiet

In [12]:
%load_ext sql
%sql sqlite:////content/exchange_analysis.db
%config SqlMagic.displaylimit = 0

In [8]:
%%sql
SELECT
    symbol,
    COUNT(*) AS row_count,
    MIN(open_time_utc) AS first_date,
    MAX(open_time_utc) AS last_date
FROM candles_daily
GROUP BY symbol;

Running query in 'sqlite:////content/exchange_analysis.db'

symbol,row_count,first_date,last_date
BTCUSDT,30,2026-09-01 00:00:00,2026-09-30 00:00:00


In [9]:
%%sql
WITH wick_metrics AS (
    SELECT
        DATE(open_time_utc) AS trading_day,
        open,
        high,
        low,
        close,
        quote_volume,
        100.0 * (MIN(open, close) - low)/ NULLIF(MIN(open, close), 0) AS lower_wick_pct,
        100.0 * (MIN(open, close) - low)/ NULLIF(high - low, 0) AS lower_wick_share_pct,
        100.0 * (open - low)/ NULLIF(open, 0) AS open_to_low_drop_pct
    FROM candles_daily
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            ORDER BY lower_wick_pct DESC, trading_day
        ) AS review_rank
    FROM wick_metrics
)
SELECT
    review_rank,
    trading_day,
    open,
    low,
    close,
    ROUND(lower_wick_pct, 2) AS lower_wick_pct,
    ROUND(lower_wick_share_pct, 2) AS lower_wick_share_pct,
    ROUND(open_to_low_drop_pct, 2) AS open_to_low_drop_pct,
    ROUND(quote_volume, 2) AS turnover_usdt
FROM ranked
WHERE review_rank <= 5
ORDER BY review_rank;

Running query in 'sqlite:////content/exchange_analysis.db'

review_rank,trading_day,open,low,close,lower_wick_pct,lower_wick_share_pct,open_to_low_drop_pct,turnover_usdt
1,2026-09-24,84397.6,82874.93,84410.24,1.8,73.65,1.8,1572042147.67
2,2026-09-02,77439.01,76264.0,77340.01,1.39,70.42,1.52,1008333112.59
3,2026-09-01,78581.3,76420.0,77439.0,1.32,36.38,2.75,1101387276.65
4,2026-09-20,81249.99,80126.04,81178.0,1.3,76.71,1.38,887777953.29
5,2026-09-22,86620.01,85114.0,86208.56,1.27,68.26,1.74,1834036114.86


In [16]:
%%sql
WITH daily AS (
    SELECT
        DATE(open_time_utc) AS trading_day,
        quote_volume AS turnover,
        trade_count,
        quote_volume / NULLIF(1.0 * trade_count, 0)
            AS avg_trade_notional
    FROM candles_daily
),
comparison AS (
    SELECT
        *,
        LAG(turnover) OVER (
            ORDER BY trading_day
        ) AS previous_turnover,

        LAG(trade_count) OVER (
            ORDER BY trading_day
        ) AS previous_trade_count,

        LAG(avg_trade_notional) OVER (
            ORDER BY trading_day
        ) AS previous_avg_notional
    FROM daily
),
changes AS (
    SELECT
        *,
        100.0 * (
            turnover / NULLIF(previous_turnover, 0) - 1
        ) AS turnover_change_pct,

        100.0 * (
            1.0 * trade_count
            / NULLIF(previous_trade_count, 0) - 1
        ) AS trade_count_change_pct,

        100.0 * (
            avg_trade_notional
            / NULLIF(previous_avg_notional, 0) - 1
        ) AS avg_notional_change_pct
    FROM comparison
),
reconciled AS (
    SELECT
        *,
        100.0 * (
            (1 + trade_count_change_pct / 100.0)
            * (1 + avg_notional_change_pct / 100.0)
            - 1
        ) AS implied_turnover_change_pct
    FROM changes
)
SELECT
    trading_day,
    ROUND(turnover_change_pct, 2)
        AS turnover_change_pct,
    ROUND(trade_count_change_pct, 2)
        AS trade_count_change_pct,
    ROUND(avg_notional_change_pct, 2)
        AS avg_notional_change_pct,
    ROUND(implied_turnover_change_pct, 2)
        AS implied_turnover_change_pct,
    ROUND(
        turnover_change_pct - implied_turnover_change_pct,
        8
    ) AS difference_pp
FROM reconciled
ORDER BY trading_day;

Running query in 'sqlite:////content/exchange_analysis.db'

trading_day,turnover_change_pct,trade_count_change_pct,avg_notional_change_pct,implied_turnover_change_pct,difference_pp
2026-09-01,None,None,None,None,None
2026-09-02,-8.45,-9.56,1.23,-8.45,0.0
2026-09-03,56.66,34.66,16.34,56.66,0.0
2026-09-04,-2.61,-23.38,27.1,-2.61,0.0
2026-09-05,-52.73,-62.67,26.61,-52.73,0.0
2026-09-06,-2.71,29.95,-25.14,-2.71,-0.0
2026-09-07,18.72,48.63,-20.12,18.72,-0.0
2026-09-08,79.8,76.08,2.11,79.8,0.0
2026-09-09,-26.18,-23.07,-4.04,-26.18,-0.0
2026-09-10,6.58,-7.72,15.5,6.58,-0.0
